# 00 — Environment setup (Colab bootstrap)

This notebook bootstraps a clean Google Colab session for the `schools-sunbeds` project. It:

1. Mounts Google Drive and creates the persistent data root at `MyDrive/schools-sunbeds-data`.
2. Clones the project repo into `/content/repo`.
3. Installs pinned dependencies from `requirements.txt` via pip.
4. Loads `.env` from Drive (gitignored, never on the public repo).
5. Verifies the manifest is intact and writes a session snapshot to `audit_logs/`.

Re-run this notebook at the start of every Colab session. Subsequent notebooks (01–14) assume `import schools_sunbeds` works and `data/manifest.csv` is intact.

## Parameters

These are the only knobs in this notebook. Edit before running.

In [ ]:
REPO_URL = "https://github.com/matthew-bowker-uos/tanning-school-walks.git"
REPO_BRANCH = "main"
DRIVE_DATA_DIR_NAME = "schools-sunbeds-data"

## 1. Mount Google Drive

In [ ]:
import os
from pathlib import Path

ON_COLAB = Path("/content").exists() and "COLAB_GPU" in os.environ or Path("/content/drive").exists()

if Path("/content").exists():
    try:
        from google.colab import drive  # type: ignore[import-not-found]
        drive.mount("/content/drive", force_remount=False)
    except ModuleNotFoundError:
        print("Not running on Colab — skipping Drive mount.")

DATA_ROOT = Path("/content/drive/MyDrive") / DRIVE_DATA_DIR_NAME
DATA_ROOT.mkdir(parents=True, exist_ok=True)
for sub in ("raw", "interim", "processed"):
    (DATA_ROOT / sub).mkdir(exist_ok=True)
print(f"DATA_ROOT = {DATA_ROOT}")

## 2. Clone the project repo and install dependencies

We install in two steps:

1. Pinned dependencies from `requirements.txt` (single source of truth, shared with local dev).
2. The project package itself, in editable mode, so `import schools_sunbeds` resolves to the cloned source tree.

In [ ]:
import subprocess
import sys

REPO_PATH = Path("/content/repo")
if REPO_PATH.exists():
    subprocess.run(["git", "-C", str(REPO_PATH), "fetch", "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "checkout", REPO_BRANCH, "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "pull", "--ff-only", "--quiet"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_PATH)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_PATH / "requirements.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPO_PATH)],
    check=True,
)
print("Repo and deps ready at", REPO_PATH)

## 3. Load `.env` from Drive

The `.env` file is **never** committed to the repo. Place it manually at `MyDrive/schools-sunbeds-data/.env` once. See `.env.example` in the repo for the schema.

In [ ]:
from dotenv import load_dotenv

ENV_PATH = DATA_ROOT / ".env"
if not ENV_PATH.exists():
    raise FileNotFoundError(
        f"Expected {ENV_PATH} — copy .env.example from the repo, fill in values, and place it in Drive."
    )

load_dotenv(ENV_PATH, override=False)
os.environ["DATA_ROOT"] = str(DATA_ROOT)

for required in ("GOOGLE_PLACES_API_KEY", "GITHUB_OWNER", "GITHUB_REPO"):
    if not os.environ.get(required):
        print(f"  MISSING: {required}")
    else:
        print(f"  set: {required}")

## 4. Verify environment

Import the package and run the manifest verifier. On a brand-new clone the manifest is empty, which verifies trivially. Once raw data has been registered (Stages 1–4) this cell becomes load-bearing: it fails closed if any registered file is missing or its hash has drifted.

In [ ]:
if str(REPO_PATH / "src") not in sys.path:
    sys.path.insert(0, str(REPO_PATH / "src"))

import schools_sunbeds
from schools_sunbeds import audit, config

config.ensure_dirs()
print("schools_sunbeds version:", schools_sunbeds.__version__)
print("DATA_ROOT          :", config.DATA_ROOT)
print("is_colab()         :", config.is_colab())
print("manifest verified  :", audit.verify_manifest() == [])

## 5. Write a session snapshot to `audit_logs/`

Each Colab session leaves a timestamped record of: package versions, hash of `requirements.txt`, git SHA of the repo, and the Python/platform string. Useful to attach to results when something looks off.

In [ ]:
import json
import platform
from datetime import datetime, timezone
from importlib.metadata import version

snapshot = {
    "session_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "platform": f"{platform.system()} {platform.machine()}",
    "git_sha": subprocess.check_output(
        ["git", "-C", str(REPO_PATH), "rev-parse", "HEAD"], text=True
    ).strip(),
    "requirements_sha256": audit.sha256_file(REPO_PATH / "requirements.txt"),
    "library_versions": {
        pkg: version(pkg)
        for pkg in (
            "geopandas",
            "shapely",
            "pyproj",
            "pandas",
            "numpy",
            "pandana",
            "osmnx",
            "networkx",
            "statsmodels",
            "scipy",
            "rapidfuzz",
        )
    },
}

snapshot_path = config.AUDIT_LOGS / f"colab_session_{snapshot['session_utc'].replace(':', '')}.json"
snapshot_path.parent.mkdir(parents=True, exist_ok=True)
snapshot_path.write_text(json.dumps(snapshot, indent=2))
print("Wrote", snapshot_path)

## Done

If the cells above all ran cleanly, you can now open any of `01_..` to `14_..` in Colab and they will pick up the same environment. Each downstream notebook starts with `audit.verify_manifest()` so manifest drift will fail loudly before any analytic work is done.